<a href="https://colab.research.google.com/github/StillWork/6FM2/blob/main/FM2_01_Claude_API_%E1%84%8E%E1%85%A2%E1%86%BA%E1%84%87%E1%85%A9%E1%86%BA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📋 claude.ai 프롬프트 / claude.ai Prompt

> **수업 활용법:** 아래 프롬프트를 [claude.ai](https://claude.ai)에 붙여넣으면 유사한 노트북을 생성할 수 있습니다.

---

### 🇰🇷 한국어 프롬프트
```
Anthropic Claude API를 활용한 두 가지 실용적인 비즈니스 AI 도구를
Google Colab 노트북으로 한국어로 구현해주세요.

[1부] 고객 서비스 챗봇:
- 시스템 프롬프트로 회사 어시스턴트 역할 정의
- 대화 이력(conversation_history 리스트)을 유지하는 멀티턴 대화
- 미리 정해진 메시지 3개로 멀티턴 대화 시뮬레이션 후,
  추가로 input()을 사용해 사용자가 직접 입력하며 챗봇과 대화하는 인터랙티브 루프 구현
  ('종료' 입력 시 루프 종료, 대화 이력 계속 유지)
- 수신 메시지를 문의/불만/칭찬/기타 유형으로 분류하는 함수

[2부] 문서 분석기:
- 비즈니스 이메일 예시 텍스트로 분석 함수 시연
- 미니 실습: google.colab.files.upload()로 PDF 파일 2개를 업로드받아
  pdfplumber로 텍스트 추출 후, 각 파일을 analyze_document()로 분석 출력

설치: anthropic, pdfplumber. 모델: claude-sonnet-4-6.
한국어 주석 포함. 초보자 친화적으로.
```

### 🇺🇸 English Prompt
```
Create a Korean Google Colab notebook with two practical business AI tools
using the Anthropic Claude API:

Part 1 — Customer Service Chatbot:
- System prompt defining a company assistant role
- Multi-turn conversation maintaining conversation_history list
- First simulate a 3-message predefined conversation, then add an interactive
  loop using input() so the user can keep chatting (type '종료' to quit),
  preserving the same conversation history throughout
- Message classifier: categorize into inquiry/complaint/compliment/other

Part 2 — Document Analyzer:
- Demo: analyze a sample business email text
- Mini exercise: use google.colab.files.upload() to upload exactly 2 PDF files,
  extract text with pdfplumber, then call analyze_document() on each and print results

Install: anthropic, pdfplumber. Model: claude-sonnet-4-6.
Korean comments. Beginner-friendly.
```

# FM2 실습 1: Claude API로 챗봇 & 문서 분석기 구현

## 학습 목표
- 시스템 프롬프트를 활용하여 AI의 역할을 정의한다
- 대화 이력을 유지하는 멀티턴 챗봇을 구현하고, 직접 입력하며 대화해본다
- Claude를 이용한 문서 분류 및 구조화된 정보 추출을 실습한다
- PDF 파일을 업로드하여 실제 문서를 AI로 분석한다

In [ ]:
!pip install anthropic pdfplumber -q
import getpass, anthropic

api_key = getpass.getpass("🔑 Anthropic API 키: ")
client = anthropic.Anthropic(api_key=api_key)
print("✅ 준비 완료!")

🔑 Anthropic API 키: ··········
✅ 준비 완료!


---
# 1부: 고객 서비스 챗봇

## 시스템 프롬프트란?

시스템 프롬프트는 **AI의 역할과 규칙을 사전에 정의**하는 메시지입니다.  
- 일반 사용자 메시지와 달리, 대화 전체에 적용됩니다
- AI의 말투, 허용 범위, 업무 도메인을 제한할 수 있습니다

In [ ]:
# 시스템 프롬프트 정의 - AI의 역할을 설정합니다
SYSTEM_PROMPT = """당신은 TechKorea 전자제품 회사의 친절한 고객 서비스 담당자입니다.

역할 및 규칙:
- 고객의 제품 관련 문의, 불만, 환불 요청을 정중하게 처리합니다
- 답변은 명확하고 공감적으로, 200자 이내로 작성합니다
- 제품과 무관한 주제는 정중히 거절하고 서비스 범위를 안내합니다
- 고객 개인정보(주민번호, 카드번호 등)는 절대 요청하지 않습니다
- 해결이 어려운 경우 '전문 상담원 연결'을 안내합니다

취급 제품: 스마트폰, 노트북, 태블릿, 스마트워치"""

# 대화 이력을 저장하는 리스트 (멀티턴 대화의 핵심)
conversation_history = []

def chat(user_message):
    """고객 메시지를 받아 챗봇 응답을 반환합니다."""
    # 사용자 메시지를 이력에 추가
    conversation_history.append({
        "role": "user",
        "content": user_message
    })

    # Claude API 호출 (시스템 프롬프트 + 전체 대화 이력 전달)
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=512,
        system=SYSTEM_PROMPT,           # 시스템 프롬프트로 역할 정의
        messages=conversation_history   # 전체 대화 이력 전달 (멀티턴)
    )

    assistant_message = response.content[0].text

    # 어시스턴트 응답도 이력에 추가 (다음 대화에서 컨텍스트로 사용)
    conversation_history.append({
        "role": "assistant",
        "content": assistant_message
    })

    return assistant_message

print("✅ 챗봇 준비 완료!")

✅ 챗봇 준비 완료!


In [ ]:
# ── 멀티턴 대화 시뮬레이션 ──────────────────────────────────
# 미리 정해진 메시지 3개로 흐름을 먼저 보여줍니다
test_messages = [
    "노트북 배터리가 갑자기 방전이 너무 빨리 되네요.",
    "어제 산 건데요, 반품이 가능한가요?",
    "반품 기간은 얼마나 되나요?"
]

for msg in test_messages:
    print(f"\n👤 고객: {msg}")
    response = chat(msg)
    print(f"🤖 상담원: {response}")

print(f"\n📊 현재 대화 이력: {len(conversation_history)}개 메시지")


👤 고객: 노트북 배터리가 갑자기 방전이 너무 빨리 되네요.
🤖 상담원: 안녕하세요! TechKorea 고객 서비스입니다. 😊

불편을 드려서 정말 죄송합니다.

**배터리 빠른 방전 시 확인 사항:**
- 🔆 화면 밝기 최대로 설정되어 있진 않나요?
- 📱 백그라운드 앱이 많이 실행 중인지 확인
- ⚡ 절전 모드 활성화 해보세요

증상이 지속된다면 **전문 상담원 연결(1588-XXXX)**을 통해 점검 도움 받으실 수 있습니다! 🙏

👤 고객: 어제 산 건데요, 반품이 가능한가요?
🤖 상담원: 구매 후 불편을 드려 정말 죄송합니다! 😔

**✅ 반품 안내**
- 구매 후 **7일 이내** 반품 가능합니다
- 제품 결함의 경우 **교환/환불** 모두 가능해요

**📋 반품 준비물**
- 구매 영수증
- 제품 및 구성품 일체

정확한 처리를 위해 **전문 상담원(1588-XXXX)**으로 연결해 드릴까요? 빠르게 도와드리겠습니다! 🙏

👤 고객: 반품 기간은 얼마나 되나요?
🤖 상담원: 안녕하세요! 😊

**📅 TechKorea 반품 기간 안내**

- **단순 변심** → 구매 후 **7일 이내**
- **제품 결함** → 구매 후 **30일 이내**

어제 구매하셨다면 충분히 반품 가능하세요! ✅

정확한 접수를 위해 **전문 상담원(1588-XXXX)** 연결을 추천드립니다. 빠르게 처리해 드리겠습니다! 🙏

📊 현재 대화 이력: 6개 메시지


## 💬 인터랙티브 챗봇: 직접 대화해보기

위에서 이어진 대화 이력을 유지한 채로 **직접 메시지를 입력**하여 챗봇과 대화합니다.  
종료하려면 `종료`, `quit`, 또는 `q`를 입력하세요.

In [ ]:
# ── 인터랙티브 대화 루프 ─────────────────────────────────────
# conversation_history는 위 시뮬레이션에서 이어집니다
print("💬 챗봇과 직접 대화를 시작합니다.")
print("   종료하려면 '종료', 'quit', 또는 'q'를 입력하세요.")
print(f"   (현재 대화 이력 {len(conversation_history)}개 메시지 유지 중)\n")

while True:
    user_input = input("👤 고객: ").strip()

    if not user_input:
        continue

    if user_input.lower() in ['종료', 'quit', 'exit', 'q']:
        print(f"\n대화를 종료합니다. 총 {len(conversation_history)}개 메시지가 기록되었습니다.")
        break

    response = chat(user_input)
    print(f"🤖 상담원: {response}\n")

💬 챗봇과 직접 대화를 시작합니다.
   종료하려면 '종료', 'quit', 또는 'q'를 입력하세요.
   (현재 대화 이력 6개 메시지 유지 중)

👤 고객: 최신 MAC 노트북 가격은 얼마나 하나요?
🤖 상담원: 안녕하세요! 😊

죄송하지만 저는 **TechKorea 제품 전담 서비스**로, 타사 제품 정보는 제공이 어렵습니다. 🙏

**TechKorea 취급 제품:**
📱 스마트폰 | 💻 노트북 | 📟 태블릿 | ⌚ 스마트워치

**TechKorea 노트북** 관련 문의는 언제든 도와드리겠습니다! 😊

👤 고객: 노트북 수리에 시간이 평균 얼마나 걸리나요?
🤖 상담원: 안녕하세요! 😊

**🔧 TechKorea 노트북 수리 기간 안내**

- **일반 수리** → 평균 **3~5 영업일**
- **부품 교체** → 평균 **5~7 영업일**
- **부품 재고 없을 시** → 최대 **2주 이상** 소요될 수 있어요

⚠️ 정확한 수리 기간은 증상에 따라 달라질 수 있습니다!

더 정확한 안내를 위해 **전문 상담원(1588-XXXX)** 연결을 추천드립니다! 🙏

👤 고객: ㅂ
🤖 상담원: 안녕하세요! 😊

문의 내용이 정확히 전달되지 않은 것 같습니다.

**TechKorea 서비스 도움 가능 항목:**
📱 스마트폰 | 💻 노트북 | 📟 태블릿 | ⌚ 스마트워치

궁금하신 점을 다시 입력해 주시면 친절히 도와드리겠습니다! 🙏

👤 고객: q

대화를 종료합니다. 총 12개 메시지가 기록되었습니다.


In [ ]:
# ── 문의 유형 자동 분류 기능 ─────────────────────────────────
def classify_message(message):
    """고객 메시지를 유형별로 분류합니다."""
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=50,
        messages=[{
            "role": "user",
            "content": f"""다음 고객 메시지를 정확히 하나의 유형으로 분류하세요.
유형: 문의 / 불만 / 칭찬 / 환불요청 / 기타
유형 이름만 답하세요.

메시지: {message}"""
        }]
    )
    return response.content[0].text.strip()

# 분류 테스트
sample_messages = [
    "스마트폰 화면이 깨졌는데 수리비가 얼마예요?",
    "제품이 도착했는데 완전히 불량품이에요! 화가 너무 납니다.",
    "상담원분이 너무 친절하게 도와주셔서 감사합니다.",
    "구매한 지 이틀밖에 안 됐는데 환불하고 싶어요."
]

print("고객 메시지 유형 분류:")
print("-" * 50)
for msg in sample_messages:
    category = classify_message(msg)
    print(f"[{category:8s}] {msg}")

고객 메시지 유형 분류:
--------------------------------------------------
[문의      ] 스마트폰 화면이 깨졌는데 수리비가 얼마예요?
[불만      ] 제품이 도착했는데 완전히 불량품이에요! 화가 너무 납니다.
[칭찬      ] 상담원분이 너무 친절하게 도와주셔서 감사합니다.
[환불요청    ] 구매한 지 이틀밖에 안 됐는데 환불하고 싶어요.


---
# 2부: 문서 분석기

이메일, 계약서, 보고서 등 비즈니스 문서에서 **핵심 정보를 자동으로 추출**합니다.

In [ ]:
def analyze_document(document_text, title="문서"):
    """비즈니스 문서를 분석하여 구조화된 정보를 추출합니다."""
    prompt = f"""다음 비즈니스 문서를 분석하여 아래 형식으로 정리해주세요:

**📌 핵심 요약 (3줄 이내):**
[핵심 내용 요약]

**✅ 액션 아이템 (해야 할 일):**
- [항목 1]
- [항목 2]

**⚠️ 주의/위험 항목:**
- [위험하거나 주의가 필요한 내용]

**📅 중요 일정/마감:**
- [날짜나 기한이 있는 항목]

문서:
{document_text}"""

    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=800,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text

print("✅ 문서 분석기 준비 완료!")

✅ 문서 분석기 준비 완료!


In [ ]:
# 테스트 문서: 비즈니스 이메일
sample_email = """
수신: 개발팀 전체
발신: 김대표 (CEO)
날짜: 2025년 3월 15일
제목: Q2 신제품 출시 일정 변경 및 긴급 대응 요청

안녕하세요, 팀 여러분.

경쟁사 A社가 당초 예정보다 한 달 앞당겨 4월 1일에 유사 제품을 출시한다는 정보를 입수했습니다.
이에 따라 우리 제품의 출시 일정을 기존 5월 15일에서 4월 20일로 앞당기기로 결정했습니다.

이를 위해 다음 사항을 즉시 진행해 주시기 바랍니다:
1. 개발팀은 3월 25일까지 베타 버전 완료 및 내부 테스트 시작
2. QA팀은 3월 26일~4월 5일 집중 테스트 진행 (주말 근무 불가피)
3. 마케팅팀은 4월 10일까지 론칭 캠페인 자료 준비 완료
4. 현재 진행 중인 기능 중 '고급 AI 추천 기능'은 이번 출시에서 제외하고 차기 버전으로 미룸

예산과 관련하여 추가 비용이 발생할 수 있으며, CFO와 협의 중입니다.
일정 변경으로 인해 현재 고용된 외주 업체와의 계약 조건 재검토가 필요합니다.

내일(3월 16일) 오전 9시에 전체 긴급 회의를 소집합니다. 반드시 참석해 주세요.
"""

print("📄 샘플 이메일 분석 결과:")
print("=" * 60)
print(analyze_document(sample_email, title="신제품 출시 일정 변경 이메일"))

📄 샘플 이메일 분석 결과:
# 📊 비즈니스 문서 분석 결과

---

## 📌 핵심 요약 (3줄 이내):
경쟁사 A社의 4월 1일 조기 출시에 대응하여, 자사 신제품 출시일을 **5월 15일 → 4월 20일로 약 25일 앞당김**. 촉박한 일정으로 인해 QA팀 주말 근무가 불가피하며, '고급 AI 추천 기능'은 이번 출시에서 제외됨. 내일 긴급 전체 회의를 통해 구체적인 대응 방안을 논의할 예정.

---

## ✅ 액션 아이템 (해야 할 일):

- **[개발팀]** 베타 버전 완료 및 내부 테스트 준비
- **[QA팀]** 집중 테스트 일정 편성 및 주말 근무 인력 조율
- **[마케팅팀]** 론칭 캠페인 자료 조기 준비 착수
- **[전 팀원]** 3월 16일 오전 9시 긴급 회의 참석
- **[경영진/담당자]** CFO와 추가 예산 협의 진행
- **[담당자]** 외주 업체 계약 조건 재검토 및 협상 진행
- **[개발팀]** '고급 AI 추천 기능' 차기 버전 로드맵으로 이관 처리

---

## ⚠️ 주의 / 위험 항목:

- 🔴 **일정 압박**: 출시일이 25일 단축되어 **품질 저하 및 개발 오류 리스크** 증가 우려
- 🔴 **주말 근무**: QA팀 주말 근무 불가피 → **팀원 번아웃 및 노동법 검토** 필요
- 🟠 **추가 비용 발생**: 예산 초과 가능성 있으며 아직 CFO 협의가 완료되지 않은 상태
- 🟠 **외주 계약 리스크**: 일정 변경으로 인한 외주 업체와의 **계약 분쟁 또는 위약금** 발생 가능
- 🟡 **기능 축소 출시**: AI 추천 기능 제외로 인한 **고객 기대치 미충족** 및 초기 반응 리스크

---

## 📅 중요 일정 / 마감:

| 날짜 | 담당 | 내용 |


## ✏️ 미니 실습: PDF 파일 2개 업로드하여 분석하기

직접 PDF 파일을 2개 업로드하면 텍스트를 추출하여 자동으로 분석합니다.

**사용 가능한 PDF:**
- 이메일 출력본, 보고서, 계약서, 회의록 등
- 한국어/영어 모두 지원

> ⚠️ 스캔 이미지로 된 PDF(텍스트 선택 불가)는 텍스트 추출이 되지 않습니다.

In [ ]:
import pdfplumber
from google.colab import files

def extract_text_from_pdf(pdf_path):
    """PDF 파일에서 전체 텍스트를 추출합니다."""
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    return text.strip()


# ── PDF 파일 2개 업로드 ────────────────────────────────────────
print("📁 PDF 파일을 2개 선택하여 업로드하세요.")
uploaded = files.upload()

pdf_files = [name for name in uploaded.keys() if name.lower().endswith('.pdf')]

if len(pdf_files) == 0:
    print("⚠️ PDF 파일이 없습니다. .pdf 파일을 업로드하세요.")
else:
    if len(pdf_files) < 2:
        print(f"⚠️ PDF 파일이 {len(pdf_files)}개만 업로드되었습니다. 2개를 권장합니다.")

    # ── 각 PDF 분석 ────────────────────────────────────────────
    for i, filename in enumerate(pdf_files[:2], start=1):
        print(f"\n{'='*65}")
        print(f"📄 파일 {i}: {filename}")
        print(f"{'='*65}")

        # 텍스트 추출
        text = extract_text_from_pdf(filename)

        if not text:
            print("❌ 텍스트를 추출할 수 없습니다. (스캔 이미지 PDF이거나 보호된 파일)")
            continue

        print(f"📝 추출된 텍스트: {len(text)}자")
        print(f"   미리보기: {text[:120].replace(chr(10), ' ')}...\n")

        # Claude로 분석 (텍스트가 너무 길면 앞 3000자만 사용)
        analysis_text = text[:3000] if len(text) > 3000 else text
        if len(text) > 3000:
            print(f"   ℹ️ 문서가 길어 앞 3000자를 분석합니다.\n")

        result = analyze_document(analysis_text, title=filename)
        print(result)

📁 PDF 파일을 2개 선택하여 업로드하세요.


Saving 수신2.pdf to 수신2.pdf
Saving 수신1.pdf to 수신1.pdf

📄 파일 1: 수신2.pdf
📝 추출된 텍스트: 442자
   미리보기: 수신: 개발팀 전체 발신: 김대표 (CEO) 날짜: 2025년 3월 15일 제목: Q2 신제품 출시 일정 변경 및 긴급 대응 요청 안녕하세요, 팀 여러분. 경쟁사 A社가 당초 예정보다 한 달 앞당겨 4월 1일에 유사...

# 📊 비즈니스 문서 분석 결과

---

## 📌 핵심 요약 (3줄 이내):
경쟁사 A社의 조기 출시(4월 1일)에 대응하여 자사 신제품 출시일을 **5월 15일 → 4월 20일**로 약 25일 앞당기기로 결정했습니다. 개발팀과 QA팀은 즉시 일정을 재편하여 베타 완료 및 집중 테스트를 수행해야 합니다. 내일 긴급 전체 회의를 통해 세부 대응 방안을 논의할 예정입니다.

---

## ✅ 액션 아이템 (해야 할 일):
- **[개발팀]** 3월 25일까지 베타 버전 완료 및 내부 테스트 착수
- **[QA팀]** 3월 26일 ~ 4월 5일 집중 테스트 수행 (주말 근무 포함)
- **[전 팀원]** 3월 16일 오전 9시 긴급 전체 회의 참석 (필수)
- **[재무/담당자]** CFO와 추가 예산 발생 관련 협의 진행
- **[계약 담당자]** 외주 업체와의 계약 조건 재검토 및 협상 진행

---

## ⚠️ 주의 / 위험 항목:
- **일정 리스크** : 기존 대비 약 25일 단축된 일정으로, 품질 저하 또는 누락 기능 발생 가능성 있음
- **인력 과부하** : QA팀 주말 근무 불가피 → 팀원 번아웃 및 업무 효율 저하 우려
- **예산 초과** : 일정 단축에 따른 추가 비용 발생 가능, 아직 CFO 승인 미확정 상태
- **계약 리스크** : 외주 업체와의 기존 계약 조건 변경 시 위약금 또는 분쟁 가능성 존재
- **경쟁사 대응** : 당사 출시일(4월 20일)이 경쟁사(4월 1일) 대비 **19일 후** → 시장 선점 효과 제한적



## 📝 정리

| 기능 | 핵심 기술 | 비즈니스 활용 |
|------|-----------|---------------|
| 멀티턴 챗봇 | `conversation_history` 누적 | 고객 서비스 자동화 |
| 인터랙티브 루프 | `input()` + `while True` | 실시간 대화형 AI |
| 시스템 프롬프트 | `system=` 파라미터 | AI 역할/범위 제한 |
| 문서 분류 | 구조화된 프롬프트 | 이메일/민원 자동 분류 |
| PDF 문서 분석 | `pdfplumber` + Claude | 보고서/계약서 검토 자동화 |

**다음 실습:** AI 에이전트 — 도구를 사용하는 더 강력한 AI 구현!